
# İran–İsrail Çatışmasında Deepfake ve Yanlış Bilgi Yayılımının Sosyal Ağ Analizi İncelenmesi

Bu çalışmada İran–İsrail çatışması bağlamında çeşitli sosyal medya platformlarından toplanan herkese açık içerikler incelenmiştir. İçerikler deepfake(yanlış bilgi), kanıtlanmış deepfake haberi, doğru bilgi ve eski bilginin yanlış bağlamda paylaşılması olmak üzere dört sınıfa ayrılmıştır. Daha sonra bu içerikler platform, hesap, hashtag ve etiket ilişkileri üzerinden sosyal ağ yapısına dönüştürülmüş; merkeziyet ölçümleri ve topluluk analizi ile yanlış bilgi yayılımının yapısı incelenmiştir.

### Proje sorusu
Bu çalışmada şu sorulara cevap aranacaktır:

- Hangi platformlarda yanlış bilgi / deepfake içerikler daha yoğun?
- Hangi hesaplar ağda daha merkezi konumda?
- Hangi hashtagler yanlış bilgiyle daha güçlü ilişkilidir?
- Ağ içinde köprü düğümler hangileridir?
- Hangi topluluklarda yanlış bilgi kümeleri yoğunlaşıyor?
- Deepfake / yanlış bilgi içerikleri mi, eski bilginin yanlış bağlamda paylaşılması mı daha yaygın?

### Kullanılan veri
Veri seti; TikTok, YouTube, Instagram, Twitter/X, Reddit ve Facebook platformlardan toplanan gönderileri içermektedir.

### Ağ modeli

**Düğüm türleri:**

- account : sosyal medya hesapları
- hashtag : gönderilerde kullanılan hashtagler
- platform : sosyal medya platformları
- label : içerik etiketleri

**Kenar türleri:**

- account ↔ account : ortak hashtag kullanan hesaplar
- account → platform : hesabın bulunduğu platform
- account → label : hesabın içerik etiketi
- account ↔ hashtag : hesabın kullandığı hashtagler

Bu yapı sayesinde hem yapısal analiz hem de profesyonel ağ görselleştirmesi yapılacaktır.


### 1. Kütüphane Kurulumu ve İçe Aktarma


In [ ]:
!pip install networkx matplotlib pandas numpy scipy python-louvain seaborn plotly -q

import warnings; warnings.filterwarnings('ignore')
import re
import os
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D
import seaborn as sns
import community.community_louvain as community_louvain
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display

BG      = '#FFFFFF'
SURFACE = '#F5F7FA'
BORDER  = '#D1D5DB'
ACCENT  = '#4F46E5'
ACCENT2 = '#EF4444'
TEXT    = '#111827'

LABEL_MAP = {
    'deepfake_yanlis_bilgi':    'Deepfake/Yanlış Bilgi',
    'dogru_bilgi':              'Doğru Bilgi',
    'eski_bilgi_yanlis_baglam': 'Eski Bilgi/Yanlış Bağlam',
    'deepfake_haberi_teyit':    'Deepfake Teyit',
}
LABEL_COLORS = {
    'deepfake_yanlis_bilgi':    '#EF4444',
    'dogru_bilgi':              '#10B981',
    'eski_bilgi_yanlis_baglam': '#F59E0B',
    'deepfake_haberi_teyit':    '#4F46E5',
}
PLATFORM_COLORS = {
    'TikTok':    '#06B6D4',
    'Instagram': '#EC4899',
    'X':         '#3B82F6',
    'Youtube':   '#EF4444',
    'Facebook':  '#6366F1',
    'Reddit':    '#F97316',
}

LAYOUT = dict(
    paper_bgcolor=BG, plot_bgcolor=SURFACE,
    font=dict(color=TEXT, family='Inter, DejaVu Sans, sans-serif', size=12),
    legend=dict(bgcolor=SURFACE, bordercolor=BORDER, borderwidth=1, font=dict(color=TEXT)),
    margin=dict(l=20, r=20, t=65, b=20),
)
AXIS_STYLE  = dict(gridcolor=BORDER, linecolor=BORDER, zerolinecolor=BORDER, tickfont=dict(color=TEXT))
AXIS_HIDDEN = dict(showgrid=False, zeroline=False, showticklabels=False)

plt.rcParams.update({
    'font.family':      'DejaVu Sans',
    'figure.dpi':       120,
    'axes.facecolor':   SURFACE,
    'figure.facecolor': BG,
    'text.color':       TEXT,
    'axes.labelcolor':  TEXT,
    'xtick.color':      TEXT,
    'ytick.color':      TEXT,
    'axes.edgecolor':   BORDER,
    'grid.color':       BORDER,
    'axes.titlecolor':  TEXT,
})

OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'✅ Hazır  |  NetworkX {nx.__version__}  |  Pandas {pd.__version__}')


### 2. Veri Yükleme ve Ön İşleme

- **Düğümler (Nodes):** Sosyal medya hesapları (içerik üreticileri)
- **Kenarlar (Edges):** Ortak hashtag kullanımı (paylaşılan etiketler aracılığıyla kurulan bağlantı)
- **Ağ türü:** Yönsüz (undirected), ağırlıklı (weighted)

In [ ]:
CSV_PATH = 'İranİsraelWarDataset.csv'
df = pd.read_csv(CSV_PATH)

NUM_COLS = ['views', 'likes', 'shares', 'comments']

for col in NUM_COLS:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.replace(',', '', regex=False)
        .str.replace('.', '', regex=False)
        .replace(['-', '', 'nan', 'None', 'none', 'null', 'Null', 'NaN'], pd.NA)
    )
    df[col] = pd.to_numeric(df[col], errors='coerce')

for col in NUM_COLS:
    def platform_zscore(x):
        sigma = x.std(skipna=True)
        if pd.isna(sigma) or sigma == 0:
            return pd.Series(pd.NA, index=x.index)
        return (x - x.mean(skipna=True)) / sigma

    df[f'{col}_z'] = df.groupby('platform')[col].transform(platform_zscore)

print('Platform bazlı z-score standardizasyon tamamlandı:')
display(df[[f'{c}_z' for c in NUM_COLS]].describe().round(3))
print()

# Eksik değer kontrolü
missing_summary = pd.DataFrame({
    'Eksik/ölçülemeyen değer': df[NUM_COLS].isna().sum(),
    'Dolu değer': df[NUM_COLS].notna().sum()
})
display(missing_summary)

display(pd.DataFrame(
    {'Değer': [len(df), df['account_name'].nunique(), len(df.columns)]},
    index=['Toplam kayıt', 'Benzersiz hesap', 'Sütun sayısı']
))
display(df.head())

### 3. Temel keşifsel analiz (EDA)
#### 3.1. Platformlara Göre Gönderi Dağılımı

In [ ]:
platform_counts = df['platform'].value_counts().reset_index()
platform_counts.columns = ['platform', 'count']

fig = px.bar(
    platform_counts, x='platform', y='count', text='count',
    color='platform', color_discrete_map=PLATFORM_COLORS,
    title='Platformlara Göre Gönderi Dağılımı',
)
fig.update_traces(textposition='outside', marker_line_width=0)
fig.update_layout(**LAYOUT, showlegend=False, height=450,
                  xaxis=dict(**AXIS_STYLE, title='Platform'),
                  yaxis=dict(**AXIS_STYLE, title='Gönderi Sayısı'))
fig.show()

#### 3.2. Etiketlere Göre İçerik Dağılımı

In [ ]:
label_counts = df['possible_label'].value_counts().reset_index()
label_counts.columns = ['possible_label', 'count']
label_counts['etiket'] = label_counts['possible_label'].map(LABEL_MAP)

fig = px.bar(
    label_counts, x='etiket', y='count', text='count',
    color='possible_label', color_discrete_map=LABEL_COLORS,
    title='Etiketlere Göre İçerik Dağılımı',
)
fig.update_traces(textposition='outside', marker_line_width=0)
fig.update_layout(**LAYOUT, showlegend=False, height=450,
                  xaxis=dict(**AXIS_STYLE, title='Etiket'),
                  yaxis=dict(**AXIS_STYLE, title='Gönderi Sayısı'))
fig.show()

#### 3.3. Platform ve Etiket İlişkisi

In [ ]:
cross_long = (
    pd.crosstab(df['platform'], df['possible_label'])
    .reset_index()
    .melt(id_vars='platform', var_name='possible_label', value_name='count')
)

fig = px.bar(
    cross_long, x='platform', y='count',
    color='possible_label', color_discrete_map=LABEL_COLORS,
    title='Platform ve Etiket İlişkisi', barmode='stack',
)
fig.update_traces(marker_line_width=0)
fig.update_layout(**LAYOUT, height=500,
                  xaxis=dict(**AXIS_STYLE, title='Platform'),
                  yaxis=dict(**AXIS_STYLE, title='Gönderi Sayısı'))
fig.show()

#### 3.4 Platform × Etiket Isı Haritası

In [ ]:
pivot = pd.crosstab(df['platform'], df['possible_label'])
pivot.columns = [LABEL_MAP.get(c, c) for c in pivot.columns]

fig = px.imshow(
    pivot, text_auto=True,
    color_continuous_scale=[[0, SURFACE], [0.5, ACCENT], [1, '#1E1B4B']],
    title='Platform × Etiket Isı Haritası', aspect='auto',
)
fig.update_layout(**LAYOUT, height=420, coloraxis_showscale=True,
                  xaxis=dict(**AXIS_STYLE, title='Etiket'),
                  yaxis=dict(**AXIS_STYLE, title='Platform'))
fig.show()

#### 3.5. En Sık Geçen 20 Hashtag

In [ ]:
def parse_hashtags(text):
    if pd.isna(text):
        return []
    tokens = re.findall(r'#?[A-Za-z0-9_İıŞşĞğÜüÖöÇç]+', str(text))
    return sorted(set(
        ('#' + t.lstrip('#')).lower() for t in tokens if len(t.lstrip('#')) > 1
    ))

df['hashtag_list'] = df['hastags'].apply(parse_hashtags) if 'hastags' in df.columns else [[] for _ in range(len(df))]
all_tags = [tag for tags in df['hashtag_list'] for tag in tags]
top20 = pd.DataFrame(Counter(all_tags).most_common(20), columns=['hashtag', 'count'])

fig = px.bar(
    top20.sort_values('count'), x='count', y='hashtag', orientation='h',
    title='En Sık Geçen 20 Hashtag',
    color='count', color_continuous_scale=[[0, SURFACE], [1, ACCENT]],
)
fig.update_layout(**LAYOUT, height=600, coloraxis_showscale=False,
                  xaxis=dict(**AXIS_STYLE, title='Kullanım Sayısı'),
                  yaxis=dict(**AXIS_STYLE, title=''))
fig.show()

### 4. Ağ Modelleme: G = (V, E, W)

**Ağ oluşturma mantığı:**
- Her hesap bir **düğüm (node)** oluşturur.
- İki hesap aynı hashtag kullandığında aralarında bir **kenar (edge)** oluşur.
- Kenar ağırlığı = ortak hashtag sayısı × iki hesabın standartlaştırılmış etkileşim skoru ortalaması.
- Bu yapı, aynı anlatıyı benimseyen hesapların kim olduğunu ortaya çıkarır.

In [ ]:
account_tags = defaultdict(set)
for _, row in df.iterrows():
    account_tags[row['account_name']].update(parse_hashtags(row['hastags'] if 'hastags' in df.columns else ''))

# 1) Standartlaştırılmış etkileşim skoru oluştur
z_cols = [f'{c}_z' for c in NUM_COLS if f'{c}_z' in df.columns]

# Her gönderi için birleşik standart skor
df['engagement_z'] = df[z_cols].mean(axis=1, skipna=True)

min_engagement_z = df['engagement_z'].min(skipna=True)
if pd.isna(min_engagement_z):
    min_engagement_z = 0
df['engagement_score'] = df['engagement_z'] - min_engagement_z + 1
df['engagement_score'] = df['engagement_score'].fillna(1)

def mode_or_first(s):
    s = s.dropna()
    if s.empty:
        return ''
    return s.mode().iloc[0] if not s.mode().empty else s.iloc[0]

def safe_sum(s):
    return s.sum(min_count=1)

account_stats = (
    df.groupby('account_name')
      .agg(
          platform=('platform', mode_or_first),
          label=('possible_label', mode_or_first),
          views=('views', safe_sum),
          likes=('likes', safe_sum),
          shares=('shares', safe_sum),
          comments=('comments', safe_sum),
          views_z=('views_z', 'mean'),
          likes_z=('likes_z', 'mean'),
          shares_z=('shares_z', 'mean'),
          comments_z=('comments_z', 'mean'),
          engagement_z=('engagement_z', 'mean'),
          engagement_score=('engagement_score', 'mean'),
      )
      .reset_index()
)

G = nx.Graph()
for _, row in account_stats.iterrows():
    acc = row['account_name']
    G.add_node(acc,
        platform=row['platform'],
        label=row['label'],
        views=float(row['views']),
        likes=float(row['likes']),
        shares=float(row['shares']),
        comments=float(row['comments']),
        views_z=float(row['views_z']),
        likes_z=float(row['likes_z']),
        shares_z=float(row['shares_z']),
        comments_z=float(row['comments_z']),
        engagement_z=float(row['engagement_z']),
        engagement_score=float(row['engagement_score']),
    )

tag_to_accounts = defaultdict(list)
for acc, tags in account_tags.items():
    if acc in G:
        for tag in tags:
            tag_to_accounts[tag].append(acc)

for accs in tag_to_accounts.values():
    for i, a1 in enumerate(accs):
        for a2 in accs[i+1:]:
            if G.has_edge(a1, a2):
                G[a1][a2]['hashtag_weight'] += 1
            else:
                G.add_edge(a1, a2, hashtag_weight=1)

for u, v, data in G.edges(data=True):
    hashtag_weight = data.get('hashtag_weight', 1)
    u_score = G.nodes[u].get('engagement_score', 1)
    v_score = G.nodes[v].get('engagement_score', 1)
    avg_std_score = (u_score + v_score) / 2

    # std_weight hesaplamalarda kullanılacak standartlaştırılmış ağırlıktır.
    data['std_weight'] = hashtag_weight * avg_std_score
    data['weight'] = data['std_weight']

    data['distance'] = 1 / data['std_weight'] if data['std_weight'] > 0 else 1

Lcc = G.subgraph(max(nx.connected_components(G), key=len)).copy()

print(f'✅ Graf oluşturuldu')
print(f'   G   → {G.number_of_nodes():,} düğüm | {G.number_of_edges():,} kenar | {nx.number_connected_components(G)} bileşen')
print(f'   LCC → {Lcc.number_of_nodes():,} düğüm | {Lcc.number_of_edges():,} kenar')
print()
print('📌 Hesaplamalarda kullanılan ağırlık: weight = std_weight')
print('   std_weight = ortak hashtag sayısı × standartlaştırılmış etkileşim skoru ortalaması')
print()
print('Platform dağılımı (G):')
print(pd.Series(nx.get_node_attributes(G, 'platform')).value_counts().to_string())

### 4.1. Ağ Temsilleri

Bu bölümde oluşturulan `G` grafiği üzerinden sosyal ağ analizi için temel gösterimler hazırlanmıştır:

- **Düğüm listesi (Node List):** Ağdaki hesapları ve düğüm özelliklerini gösterir.
- **Kenar listesi (Edge List):** Hesaplar arasındaki ortak hashtag bağlantılarını ve ağırlıkları gösterir.
- **Komşuluk matrisi (Adjacency Matrix):** Düğümler arasındaki bağlantıları matris biçiminde gösterir.
- **Komşuluk listesi (Adjacency List):** Her düğümün doğrudan bağlı olduğu komşuları listeler.
- **NetworkX ile Ağ Gösterimi:** Düğümler, kenarlar, bağlantı ağırlıkları ve komşuluk ilişkileriyle birlikte modeller.

In [ ]:
from pathlib import Path
from IPython.display import display

print(f'Kullanılan ağ: LCC | Düğüm: {Lcc.number_of_nodes()} | Kenar: {Lcc.number_of_edges()}')

def fmt_num(x):
    if pd.isna(x):
        return "Veri yok"
    return f"{int(x):,}"

def fmt_z(x):
    if pd.isna(x):
        return "NaN"
    return f"{x:.2f}"

# 1) DÜĞÜM LİSTESİ
node_list = pd.DataFrame([
    {
        'node': node,
        'platform': data.get('platform', ''),
        'label': data.get('label', ''),
        'label_readable': LABEL_MAP.get(data.get('label', ''), data.get('label', '')),
        'views': data.get('views'),
        'likes': data.get('likes'),
        'shares': data.get('shares'),
        'views_z': data.get('views_z'),
        'likes_z': data.get('likes_z'),
        'shares_z': data.get('shares_z'),
        'comments_z': data.get('comments_z'),
        'engagement_z': data.get('engagement_z', 0),
        'engagement_score': data.get('engagement_score', 0),
        'degree': Lcc.degree(node),
        'weighted_degree_std': Lcc.degree(node, weight='weight')
    }
    for node, data in Lcc.nodes(data=True)
])

print('\nDüğüm Listesi (İlk 10 Satır)')
display(node_list.head(10))
node_list.to_csv(OUTPUT_DIR / 'node_list.csv', index=False, encoding='utf-8-sig')

# 2) KENAR LİSTESİ
edge_list = pd.DataFrame([
    {
        'source': u,
        'target': v,
        'hashtag_weight': data.get('hashtag_weight', 1),
        'std_weight': data.get('std_weight', data.get('weight', 1))
    }
    for u, v, data in Lcc.edges(data=True)
])

print('\nKenar Listesi (İlk 10 Satır)')
display(edge_list.head(10))
edge_list.to_csv(OUTPUT_DIR / 'edge_list.csv', index=False, encoding='utf-8-sig')

# 3) KOMŞULUK MATRİSİ
adjacency_matrix = nx.to_pandas_adjacency(Lcc, weight='weight', dtype=float)

print('\nKomşuluk Matrisi (İlk 10×10)')
display(adjacency_matrix.iloc[:10, :10])
adjacency_matrix.to_csv(OUTPUT_DIR / 'adjacency_matrix.csv', encoding='utf-8-sig')

# 4) KOMŞULUK LİSTESİ
adjacency_list = pd.DataFrame([
    {
        'node': node,
        'neighbors': ', '.join(list(Lcc.neighbors(node))),
        'neighbor_count': len(list(Lcc.neighbors(node)))
    }
    for node in Lcc.nodes()
])

print('\nKomşuluk Listesi (İlk 10 Satır)')
display(adjacency_list.head(10))
adjacency_list.to_csv(OUTPUT_DIR / 'adjacency_list.csv', index=False, encoding='utf-8-sig')

# 5) NETWORKX İLE AĞ GÖSTERİMİ
pos_network = nx.spring_layout(Lcc, seed=42, k=0.35)

edge_x, edge_y = [], []

for u, v in Lcc.edges():
    x0, y0 = pos_network[u]
    x1, y1 = pos_network[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    mode='lines',
    line=dict(width=0.6, color='#555555'),
    opacity=0.3,
    hoverinfo='none',
    showlegend=False
)

degree_dict = dict(Lcc.degree())
max_degree = max(degree_dict.values()) if len(degree_dict) > 0 and max(degree_dict.values()) > 0 else 1

platform_traces = []

for platform, color in PLATFORM_COLORS.items():
    platform_nodes = [
        n for n in Lcc.nodes()
        if Lcc.nodes[n].get('platform') == platform
    ]

    if not platform_nodes:
        continue

    hover_texts = [
        f"<b>👤 {n}</b><br>"
        f"📱 Platform: {Lcc.nodes[n].get('platform', '?')}<br>"
        f"🏷️ Etiket: {LABEL_MAP.get(Lcc.nodes[n].get('label', ''), '?')}<br>"
        f"👁️ Görüntülenme: {fmt_num(Lcc.nodes[n].get('views'))} "
        f"<i>(z={fmt_z(Lcc.nodes[n].get('views_z'))})</i><br>"
        f"❤️ Beğeni: {fmt_num(Lcc.nodes[n].get('likes'))} "
        f"<i>(z={fmt_z(Lcc.nodes[n].get('likes_z'))})</i><br>"
        f"🔗 Derece: {Lcc.degree(n)}<br>"
        f"⚖️ Standartlaştırılmış ağırlıklı derece: {Lcc.degree(n, weight='weight'):.2f}<br>"
        f"📊 Etkileşim z-skoru: {fmt_z(Lcc.nodes[n].get('engagement_z'))}"
        for n in platform_nodes
    ]

    node_sizes = [
        5 + 35 * (Lcc.degree(n) / max_degree) ** 0.4
        for n in platform_nodes
    ]

    platform_traces.append(go.Scatter(
        x=[pos_network[n][0] for n in platform_nodes],
        y=[pos_network[n][1] for n in platform_nodes],
        mode='markers',
        name=platform,
        marker=dict(
            size=node_sizes,
            color=color,
            opacity=0.88,
            line=dict(width=0.5, color='white')
        ),
        text=hover_texts,
        hovertemplate='%{text}<extra></extra>'
    ))

fig_network = go.Figure(data=[edge_trace] + platform_traces)

fig_network.update_layout(
    **LAYOUT,
    hovermode="closest",
    height=750,
    title=dict(
        text="NetworkX ile Ağ Gösterimi<br>"
             "<sup>Düğüm = Hesap | Kenar = Ortak Hashtag | Renk = Platform | Üzerine gelin → z-score dahil detay</sup>",
        font=dict(color=TEXT, size=14)
    ),
    xaxis=AXIS_HIDDEN,
    yaxis=AXIS_HIDDEN
)

fig_network.show()



### 5. Temel Ağ Ölçütleri

Bu ölçütler ağın genel yapısını, sıklığını ve bilgi iletim kapasitesini ortaya koyar.

- **Node count / edge count:** Ağın büyüklüğünü gösterir.
- **Density:** Ağın genel bağlantı yoğunluğunu gösterir.
- **Average degree:** Ortalama bir düğümün kaç bağlantısı olduğunu gösterir.
- **Connected components:** Ağın parçalı olup olmadığını gösterir.
- **Clustering coefficient:** Düğümlerin yerel kümelenme eğilimini gösterir.
- **Diameter:** En büyük bileşendeki iki düğüm arasındaki en uzun en kısa yol.

In [ ]:
components = list(nx.connected_components(G))

avg_degree = np.mean([d for _, d in G.degree()])
avg_clust  = nx.average_clustering(G)
diameter   = nx.diameter(Lcc) if len(Lcc) > 1 else 0
avg_sp     = nx.average_shortest_path_length(Lcc) if len(Lcc) > 1 else 0

display(pd.DataFrame({
    'Ölçüt': [
        'Düğüm sayısı', 'Kenar sayısı', 'Yoğunluk', 'Ortalama derece',
        'Ortalama kümelenme kat.', 'Bileşen sayısı',
        'En büyük bileşen (düğüm)', 'Çap (LCC)', 'Ort. en kısa yol (LCC)',
    ],
    'Değer': [
        G.number_of_nodes(), G.number_of_edges(),
        round(nx.density(G), 6), round(avg_degree, 3),
        round(avg_clust, 6), len(components),
        len(max(components, key=len)), diameter, round(avg_sp, 4),
    ]
}))


### 6. Derece Dağılımı Analizi

In [ ]:
from plotly.subplots import make_subplots

degrees     = [d for _, d in G.degree()]
avg_deg     = np.mean(degrees)
deg_counter = Counter(degrees)
k_vals      = sorted(deg_counter)
p_vals      = [deg_counter[k] / len(degrees) for k in k_vals]

fig_deg = make_subplots(rows=1, cols=2,
    subplot_titles=["Derece Histogramı", "Log-Log Derece Dağılımı"],
    horizontal_spacing=0.1)

# Histogram
fig_deg.add_trace(go.Histogram(
    x=degrees, nbinsx=50, name="Frekans",
    marker=dict(color=ACCENT, line=dict(width=0.3, color="#FFFFFF")),
    opacity=0.9
), row=1, col=1)
fig_deg.add_vline(x=avg_deg, line_dash="dash", line_color=ACCENT2,
    annotation_text=f"Ort. = {avg_deg:.1f}", annotation_font_color=ACCENT2, row=1, col=1)

# Log-log scatter
fig_deg.add_trace(go.Scatter(
    x=k_vals, y=p_vals, mode="markers", name="P(k)",
    marker=dict(size=6, color=ACCENT2, opacity=0.8)
), row=1, col=2)

fig_deg.update_layout(
    **LAYOUT, height=480, showlegend=False,
    title=dict(text="Derece Dağılımı Analizi", font=dict(color=TEXT, size=14)),
)
fig_deg.update_xaxes(title_text="Derece", row=1, col=1, **AXIS_STYLE)
fig_deg.update_yaxes(title_text="Frekans (log)", type="log", row=1, col=1, **AXIS_STYLE)
fig_deg.update_xaxes(title_text="Derece k (log)", type="log", row=1, col=2, **AXIS_STYLE)
fig_deg.update_yaxes(title_text="P(k) (log)", type="log", row=1, col=2, **AXIS_STYLE)
fig_deg.update_annotations(font=dict(color=TEXT, size=11))
fig_deg.show()


### 7. Merkezilik Analizi

Dört farklı merkezilik ölçütü, ağdaki düğümlerin farklı boyutlardaki önemini ortaya koyar:

| Ölçüt | Anlamı |
|-------|--------|
| **Degree Centrality** | En çok bağlantıya sahip (popüler) hesap |
| **Betweenness Centrality** | Bilgi akışında köprü rolündeki hesap |
| **Closeness Centrality** | Ağa en hızlı erişen, en hızlı yayıcı |
| **Eigenvector Centrality** | Etkili hesaplara bağlı, dolayısıyla güçlü hesap |
| **Page Rank** | Otorite/önem |

In [ ]:
print('⏳ Merkezilik hesaplanıyor...')
node_attr = dict(Lcc.nodes(data=True))

centralities = {
    'Degree':      (nx.degree_centrality(Lcc),                                      'Bağlantı sayısına göre en merkezi hesaplar'),
    'Weighted Degree': ({n: Lcc.degree(n, weight='weight') for n in Lcc.nodes()},    'Standartlaştırılmış etkileşim ağırlığına göre güçlü hesaplar'),
    'Betweenness': (nx.betweenness_centrality(Lcc, normalized=True, weight='distance'), 'Standart ağırlıklı mesafeye göre köprü hesaplar'),
    'Closeness':   (nx.closeness_centrality(Lcc, distance='distance'),              'Standart ağırlıklı mesafeyle ağa en yakın hesaplar'),
    'Eigenvector': (nx.eigenvector_centrality(Lcc, max_iter=1000, weight='weight'),  'Standart ağırlıklı güçlü hesaplarla bağlantılı hesaplar'),
    'PageRank':    (nx.pagerank(Lcc, weight='weight'),                              'Standartlaştırılmış ağırlıklı bağlantıya göre otorite hesaplar'),
}
print('✅ Tamamlandı\n')

deg_cen   = centralities['Degree'][0]
wdeg_cen  = centralities['Weighted Degree'][0]
bet_cen   = centralities['Betweenness'][0]
clo_cen   = centralities['Closeness'][0]
eig_cen   = centralities['Eigenvector'][0]
pr_cen    = centralities['PageRank'][0]

for name, (cen, desc) in centralities.items():
    print(f'  {"─"*65}')
    print(f'  {name:<15} → {desc}')
    print(f'  {"─"*65}')
    print(f'  {"Sıra":<5} {"Hesap":<28} {"Değer":<10} {"Platform":<12} Etiket')
    for i, (node, val) in enumerate(sorted(cen.items(), key=lambda x: -x[1])[:5], 1):
        attr = node_attr.get(node, {})
        lbl  = LABEL_MAP.get(attr.get('label', ''), '?')
        print(f'  {i:<5} {node.strip():<28} {val:.4f}     {attr.get("platform","?"):<12} {lbl}')
    print()

degree_c      = deg_cen
betweenness_c = bet_cen
closeness_c   = clo_cen
pagerank_c    = pr_cen

cent_df = pd.DataFrame({
    'Hesap': list(deg_cen.keys()),
    'Degree': list(deg_cen.values()),
    'Weighted_Degree_Std': [wdeg_cen.get(n, 0) for n in deg_cen],
    'Betweenness_Weighted': [bet_cen.get(n, 0) for n in deg_cen],
    'Closeness_Weighted': [clo_cen.get(n, 0) for n in deg_cen],
    'PageRank_Weighted': [pr_cen.get(n, 0) for n in deg_cen],
    'Engagement_Z': [Lcc.nodes[n].get('engagement_z', 0) for n in deg_cen],
    'Engagement_Score': [Lcc.nodes[n].get('engagement_score', 0) for n in deg_cen],
}).sort_values('Weighted_Degree_Std', ascending=False).reset_index(drop=True)

display(cent_df.head(10))


### 8. Topluluk Analizi (Louvain Algoritması)

Louvain algoritması, ağı yoğun iç bağlantılara sahip gruplar (topluluklar) halinde böler. Modülerlik değeri ne kadar yüksekse topluluk yapısı o kadar belirgindir.

In [ ]:
partition  = community_louvain.best_partition(Lcc, weight='weight', random_state=42)
modularity = community_louvain.modularity(partition, Lcc)
comm_ids   = set(partition.values())

community_info = {}
for cid in comm_ids:
    members = [n for n, c in partition.items() if c == cid]
    plats   = Counter(Lcc.nodes[n]['platform'] for n in members)
    labels  = Counter(Lcc.nodes[n]['label']    for n in members)
    community_info[cid] = {
        'size':           len(members),
        'top_platform':   plats.most_common(1)[0][0],
        'top_label':      LABEL_MAP.get(labels.most_common(1)[0][0], '?'),
        'deepfake_ratio': labels.get('deepfake_yanlis_bilgi', 0) / len(members),
    }

print(f'  Topluluk sayısı : {len(comm_ids)}')
print(f'  Modülerlik Q   : {modularity:.4f}  (>0.3 → güçlü topluluk yapısı)')
print()
print(f'  {"ID":<5} {"Boyut":<8} {"Baskın Platform":<18} {"Baskın Etiket":<30} Deepfake Oranı')
print(f'  {"─"*78}')
for cid, info in sorted(community_info.items(), key=lambda x: -x[1]['size']):
    print(f'  {cid:<5} {info["size"]:<8} {info["top_platform"]:<18} {info["top_label"]:<30} %{info["deepfake_ratio"]*100:.1f}')

bridge_nodes = sorted(
    [(n, len(set(partition[nb] for nb in Lcc.neighbors(n))))
     for n in Lcc.nodes() if len(set(partition[nb] for nb in Lcc.neighbors(n))) > 1],
    key=lambda x: -x[1]
)
print(f'\n  Köprü Düğümler (Top 5):')
for node, nc in bridge_nodes[:5]:
    attr = Lcc.nodes[node]
    print(f'  {node.strip():<32} {nc} topluluk | {attr["platform"]} | {LABEL_MAP.get(attr["label"],"?")}')

for n, c in partition.items():
    if n in G:
        G.nodes[n]['community'] = c


### 9. Görselleştirmeler

#### 9.1 Genel Ağ Grafiği – Platform ve İçerik Türüne Göre

Düğüm renkleri içerik türünü (doğru bilgi, deepfake vb.), düğüm boyutları ise görüntülenme sayısını temsil etmektedir.

In [ ]:
def fmt_num(x):
    if pd.isna(x):
        return "Veri yok"
    return f"{int(x):,}"

def fmt_z(x):
    if pd.isna(x):
        return "NaN"
    return f"{x:.2f}"

pos     = nx.spring_layout(Lcc, seed=42, k=0.3)
max_std = max((Lcc.nodes[n].get('engagement_score', 1) for n in Lcc.nodes()), default=1) or 1

edge_x, edge_y = [], []
for u, v in Lcc.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]; edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.5, color='#555555'),
    hoverinfo='none', showlegend=False,
)

GRAPH_LAYOUT = dict(
    **LAYOUT, hovermode='closest', height=700,
    xaxis=AXIS_HIDDEN, yaxis=AXIS_HIDDEN,
)

def node_size(n):
    return 8 + 32 * (Lcc.nodes[n].get('engagement_score', 1) / max_std) ** 0.4

traces = []
for plat, color in PLATFORM_COLORS.items():
    nodes = [n for n in Lcc.nodes() if Lcc.nodes[n].get('platform') == plat]
    if not nodes: continue
    hover = [
        f"<b>👤 {n}</b><br>"
        f"📱 {Lcc.nodes[n].get('platform','?')}<br>"
        f"🏷️ {LABEL_MAP.get(Lcc.nodes[n].get('label',''),'?')}<br>"
        f"👁️ Görüntülenme: {fmt_num(Lcc.nodes[n].get('views'))} "
        f"<i>(z={fmt_z(Lcc.nodes[n].get('views_z'))})</i><br>"
        f"❤️ Beğeni: {fmt_num(Lcc.nodes[n].get('likes'))} "
        f"<i>(z={fmt_z(Lcc.nodes[n].get('likes_z'))})</i><br>"
        f"🔗 Derece: {Lcc.degree(n)}<br>"
        f"⚖️ Std. ağırlıklı derece: {Lcc.degree(n, weight='weight'):.2f}<br>"
        f"📊 Etkileşim z: {Lcc.nodes[n].get('engagement_z'):.2f}"
        for n in nodes
    ]
    traces.append(go.Scatter(
        x=[pos[n][0] for n in nodes], y=[pos[n][1] for n in nodes],
        mode='markers', name=plat,
        marker=dict(size=[node_size(n) for n in nodes], color=color,
                    opacity=0.88, line=dict(width=0.5, color='white')),
        text=hover, hovertemplate='%{text}<extra></extra>',
    ))

fig_plat = go.Figure(data=[edge_trace] + traces)
fig_plat.update_layout(**GRAPH_LAYOUT,
                        title=dict(text='Platform Bazlı Ağ<br><sup>Düğüm boyutu = Standartlaştırılmış etkileşim skoru | Üzerine gelin → z-score dahil detay</sup>',
                                   font=dict(color=TEXT, size=14)))
fig_plat.show()

# Etiket bazlı ağ
traces2 = []
for lbl_key, color in LABEL_COLORS.items():
    nodes = [n for n in Lcc.nodes() if Lcc.nodes[n].get('label') == lbl_key]
    if not nodes: continue
    hover = [
        f"<b>👤 {n}</b><br>"
        f"📱 {Lcc.nodes[n].get('platform','?')}<br>"
        f"🏷️ {LABEL_MAP.get(Lcc.nodes[n].get('label',''),'?')}<br>"
        f"🔁 Paylaşım: {fmt_num(Lcc.nodes[n].get('shares'))} "
        f"<i>(z={fmt_z(Lcc.nodes[n].get('shares_z'))})</i><br>"
        f"🔗 Derece: {Lcc.degree(n)}<br>"
        f"⚖️ Std. ağırlıklı derece: {Lcc.degree(n, weight='weight'):.2f}<br>"
        f"📊 Etkileşim z: {fmt_z(Lcc.nodes[n].get('engagement_z'))}"
        for n in nodes
    ]
    traces2.append(go.Scatter(
        x=[pos[n][0] for n in nodes], y=[pos[n][1] for n in nodes],
        mode='markers', name=LABEL_MAP.get(lbl_key, lbl_key),
        marker=dict(size=[node_size(n) for n in nodes], color=color,
                    opacity=0.88, line=dict(width=0.5, color='white')),
        text=hover, hovertemplate='%{text}<extra></extra>',
    ))

fig_lbl = go.Figure(data=[edge_trace] + traces2)
fig_lbl.update_layout(**GRAPH_LAYOUT,
                       title=dict(text='İçerik Etiketi Bazlı Ağ<br><sup>Üzerine gelin → z-score dahil detay</sup>',
                                  font=dict(color=TEXT, size=14)))
fig_lbl.show()


### 9.2 Merkezilik Bazlı Ağ Grafiği

Düğüm boyutu ve rengi, dört farklı merkezilik ölçütünü yansıtır. Bu görsel "kim önemli, kim köprü, kim hızlı yayıcı?" sorularını görsel olarak yanıtlar.

In [ ]:
# Merkezilik Bazlı Ağ Grafiği – 2x2 Panel

from plotly.subplots import make_subplots

centrality_configs = [
    ('Degree Centrality',      deg_cen,  'Plasma',   'En popüler hesaplar'),
    ('Betweenness Centrality', bet_cen,  'Inferno',  'Köprü hesaplar'),
    ('Closeness Centrality',   clo_cen,  'Viridis',  'En hızlı yayıcılar'),
    ('Eigenvector Centrality', eig_cen,  'Magma',    'En etkili hesaplar'),
]

fig3 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[f"{name} – {desc}" for name, _, _, desc in centrality_configs],
    horizontal_spacing=0.06, vertical_spacing=0.12
)

panel_positions = [(1,1), (1,2), (2,1), (2,2)]

for (row, col), (cen_name, centrality, colorscale, desc) in zip(panel_positions, centrality_configs):
    # Kenarlar
    ex, ey = [], []
    for u, v in Lcc.edges():
        x0, y0 = pos[u]; x1, y1 = pos[v]
        ex += [x0, x1, None]; ey += [y0, y1, None]
    fig3.add_trace(go.Scatter(
        x=ex, y=ey, mode='lines',
        line=dict(width=0.5, color='#555555'),
        hoverinfo='none', showlegend=False
    ), row=row, col=col)

    # Düğümler
    nodes = list(Lcc.nodes())
    cen_vals = [centrality.get(n, 0) for n in nodes]
    max_cen = max(cen_vals) if max(cen_vals) > 0 else 1
    norm_cen = [v / max_cen for v in cen_vals]
    node_sz = [6 + 38 * (v ** 0.5) for v in norm_cen]
    xs = [pos[n][0] for n in nodes]
    ys = [pos[n][1] for n in nodes]

    hover_texts = []
    for n, cval, nval in zip(nodes, cen_vals, norm_cen):
        attr = Lcc.nodes[n]
        lbl = LABEL_MAP.get(attr.get('label', ''), '?')
        hover_texts.append(
            f"<b>👤 {n}</b><br>"
            f"📊 {cen_name}: {cval:.4f}<br>"
            f"📈 Normalize: {nval:.3f}<br>"
            f"📱 Platform: {attr.get('platform','?')}<br>"
            f"🏷️ Etiket: {lbl}<br>"
            f"👁️ Görüntülenme: {fmt_num(attr.get('views'))}"
        )

    # Colorbar yalnızca sol panelde (col==1) göster
    show_colorbar = (col == 1)
    fig3.add_trace(go.Scatter(
        x=xs, y=ys, mode='markers',
        marker=dict(
            size=node_sz, color=norm_cen, colorscale=colorscale,
            showscale=show_colorbar,
            colorbar=dict(
                title=dict(text="Merkezilik", font=dict(color=TEXT, size=9)),
                tickfont=dict(color=TEXT, size=8), len=0.38,
                x=-0.02 if col == 1 else None,
                y=0.78 if row == 1 else 0.22
            ) if show_colorbar else {},
            opacity=0.85, line=dict(width=0)
        ),
        text=hover_texts,
        hovertemplate="%{text}<extra></extra>",
        showlegend=False
    ), row=row, col=col)

fig3.update_layout(
    title=dict(
        text="Merkezilik Ölçütlerine Göre Ağ Görselleştirmesi<br>"
             "<sup>Düğüm üzerine gelin → hesap adı, merkezilik değeri ve detaylar</sup>",
        font=dict(color=TEXT, size=15)
    ),
    paper_bgcolor=BG, plot_bgcolor=BG,
    font=dict(color=TEXT), hovermode='closest',
    height=1000, margin=dict(l=60, r=30, t=100, b=20)
)
for i in range(1, 5):
    r = (i - 1) // 2 + 1; c = (i - 1) % 2 + 1
    fig3.update_xaxes(showgrid=False, zeroline=False, showticklabels=False, row=r, col=c)
    fig3.update_yaxes(showgrid=False, zeroline=False, showticklabels=False, row=r, col=c)
fig3.update_annotations(font=dict(color=TEXT, size=11))
fig3.show()


### 9.3 Topluluk Analizi Görselleştirmesi

Her renk bir topluluğu temsil eder. Aynı renkteki hesaplar benzer hashtag örüntüleriyle birbirine bağlı gruplardır.

In [ ]:
COMM_COLORS = [ACCENT, ACCENT2, '#10B981', '#F59E0B', '#8B5CF6', '#06B6D4', '#EC4899', '#14B8A6']

def safe_node_score(n):
    score = Lcc.nodes[n].get('engagement_score', 1)
    return 1 if pd.isna(score) else score

node_scores = [safe_node_score(n) for n in Lcc.nodes()]
max_v = max(node_scores) if len(node_scores) > 0 and max(node_scores) > 0 else 1

comm_members = {}
for cid in set(partition.values()):
    comm_members[cid] = [n for n, c in partition.items() if c == cid]

edge_x2, edge_y2 = [], []
for u, v in Lcc.edges():
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x2 += [x0, x1, None]; edge_y2 += [y0, y1, None]

edge_tr = go.Scatter(
    x=edge_x2, y=edge_y2, mode='lines',
    line=dict(width=0.5, color='#555555'),
    hoverinfo='none', showlegend=False
)

comm_traces = []
for comm_id in sorted(set(partition.values())):
    members = comm_members[comm_id]
    color   = COMM_COLORS[comm_id % len(COMM_COLORS)]
    info    = community_info[comm_id]
    xs = [pos[n][0] for n in members]
    ys = [pos[n][1] for n in members]
    sizes = [
    8 + 25 * (safe_node_score(n) / max_v) ** 0.4
    for n in members]
    hover_texts = [
        f"<b>👤 {n}</b><br>"
        f"🏘️ Topluluk: T{comm_id}<br>"
        f"📌 Baskın Platform: {info['top_platform']}<br>"
        f"📱 Hesap Platformu: {Lcc.nodes[n].get('platform','?')}<br>"
        f"👥 Boyut: {info['size']} üye<br>"
        f"🏷️ Etiket: {LABEL_MAP.get(Lcc.nodes[n].get('label',''), '?')}<br>"
        f"⚠️ Deepfake oranı: %{info['deepfake_ratio']*100:.1f}<br>"
        f"👁️ Görüntülenme: {fmt_num(Lcc.nodes[n].get('views'))} (z={fmt_z(Lcc.nodes[n].get('views_z'))})"
        for n in members
    ]
    comm_traces.append(go.Scatter(
        x=xs, y=ys, mode='markers',
        name=f"T{comm_id} – {info['top_platform']} (n={info['size']})",
        marker=dict(size=sizes, color=color, opacity=0.85, line=dict(width=0.4, color='white')),
        text=hover_texts, hovertemplate='%{text}<extra></extra>'
    ))

fig4 = go.Figure(
    data=[edge_tr] + comm_traces,
    layout=go.Layout(
        **LAYOUT, hovermode='closest', height=750,
        title=dict(text='Louvain Topluluk Analizi<br><sup>Renk = Topluluk | Üzerine gelin → detay ve deepfake oranı</sup>',
                   font=dict(color=TEXT, size=14)),
        xaxis=AXIS_HIDDEN, yaxis=AXIS_HIDDEN,
    )
)
fig4.show()

# Topluluk deepfake oranı çubuğu
comm_ids_sorted = sorted(community_info.keys(), key=lambda x: -community_info[x]['deepfake_ratio'])
bar_labels = [f"T{c} ({community_info[c]['top_platform']}, n={community_info[c]['size']})" for c in comm_ids_sorted]
df_ratios  = [community_info[c]['deepfake_ratio'] * 100 for c in comm_ids_sorted]
bar_colors = [COMM_COLORS[c % len(COMM_COLORS)] for c in comm_ids_sorted]

fig4b = go.Figure(go.Bar(
    x=df_ratios, y=bar_labels, orientation='h',
    marker=dict(color=bar_colors, line=dict(color='white', width=0.5)),
    text=[f"%{v:.1f}" for v in df_ratios], textposition='outside',
    hovertemplate="<b>%{y}</b><br>Deepfake Oranı: %{x:.1f}%<extra></extra>"
))
fig4b.add_vline(x=25, line_dash='dash', line_color=ACCENT2,
                annotation_text='Referans %25', annotation_font_color=ACCENT2)
fig4b.update_layout(
    **LAYOUT, height=500,
    title=dict(text='Topluluk Başına Deepfake İçerik Oranı', font=dict(color=TEXT, size=13)),
    xaxis=dict(**AXIS_STYLE, title='Deepfake/Yanlış Bilgi İçerik Oranı (%)'),
    yaxis=dict(**AXIS_STYLE),
)
fig4b.show()


#### 9.4 Genel Ağ Grafiği

In [ ]:
top_nodes = cent_df.nlargest(600, 'Weighted_Degree_Std')['Hesap'].tolist() if 'cent_df' in dir() and 'Weighted_Degree_Std' in cent_df.columns else list(Lcc.nodes())[:600]
G_vis = Lcc.subgraph(top_nodes).copy() if len(Lcc) > 600 else Lcc.copy()

np.random.seed(42)
pos_vis = nx.spring_layout(G_vis, k=0.8, iterations=80, seed=42)

ex36, ey36 = [], []
for u, v in G_vis.edges():
    x0, y0 = pos_vis[u]; x1, y1 = pos_vis[v]
    ex36 += [x0, x1, None]; ey36 += [y0, y1, None]
edge_tr36 = go.Scatter(
    x=ex36, y=ey36, mode='lines',
    line=dict(width=0.4, color='#555555'),
    hoverinfo='none', showlegend=False, opacity=0.15
)

plat_traces36 = []
for plat, color in PLATFORM_COLORS.items():
    nodes36 = [n for n in G_vis.nodes() if G_vis.nodes[n].get('platform') == plat]
    if not nodes36: continue
    hover36 = [
        f"<b>👤 {n}</b><br>"
        f"📱 {G_vis.nodes[n].get('platform','?')}<br>"
        f"🏷️ {LABEL_MAP.get(G_vis.nodes[n].get('label',''),'?')}<br>"
        f"🔗 Degree: {degree_c.get(n,0):.4f}<br>"
        f"⚖️ Weighted degree std: {Lcc.degree(n, weight='weight'):.2f}<br>"
        f"📊 Engagement z: {Lcc.nodes[n].get('engagement_z',0):.2f}"
        for n in nodes36
    ]
    plat_traces36.append(go.Scatter(
        x=[pos_vis[n][0] for n in nodes36], y=[pos_vis[n][1] for n in nodes36],
        mode='markers', name=plat,
        marker=dict(size=[5 + 25 * (Lcc.degree(n, weight='weight') / max(dict(Lcc.degree(weight='weight')).values())) for n in nodes36],
                    color=color, opacity=0.85, line=dict(width=0)),
        text=hover36, hovertemplate='%{text}<extra></extra>'
    ))

fig36 = go.Figure(data=[edge_tr36] + plat_traces36)
fig36.update_layout(
    **LAYOUT, hovermode="closest", height=750,
    title=dict(
        text="İran-İsrail Çatışması: Sosyal Medya Hesap Ağı<br>"
             "<sup>Her düğüm = bir hesap | Renk = Platform | Boyut = Standartlaştırılmış ağırlıklı derece | Üzerine gelin → detay</sup>",
        font=dict(color=TEXT, size=14)
    ),
    xaxis=AXIS_HIDDEN, yaxis=AXIS_HIDDEN
)
fig36.show()

#### 9.5 Yanlış Bilgi vs Doğru Bilgi Ağı

In [ ]:
from plotly.subplots import make_subplots

# is_misinfo flag ekle
misinfo_labels = {'deepfake_yanlis_bilgi', 'eski_bilgi_yanlis_baglam'}
for n in G_vis.nodes():
    G_vis.nodes[n]['is_misinfo'] = G_vis.nodes[n].get('label', '') in misinfo_labels

fig38 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["⚠️ YANLIŞ BİLGİ / DEEPFAKE Hesapları", "✅ DOĞRU BİLGİ Hesapları"],
    horizontal_spacing=0.06
)

# Arka plan tüm düğümler (gri)
all_x38 = [pos_vis[n][0] for n in G_vis.nodes()]
all_y38 = [pos_vis[n][1] for n in G_vis.nodes()]
ex38, ey38 = [], []
for u, v in G_vis.edges():
    x0, y0 = pos_vis[u]; x1, y1 = pos_vis[v]
    ex38 += [x0, x1, None]; ey38 += [y0, y1, None]

for col, (is_mis, hi_color, edge_hi) in enumerate([
    (True,  ACCENT2,  '#FCA5A5'),
    (False, '#10B981', '#6EE7B7')
], 1):
    # Arka plan
    fig38.add_trace(go.Scatter(
        x=all_x38, y=all_y38,
        mode='markers',
        marker=dict(size=4, color='#D1D5DB', opacity=0.25),
        hoverinfo='none', showlegend=False
    ), row=1, col=col)
    fig38.add_trace(go.Scatter(
        x=ex38, y=ey38,
        mode='lines',
        line=dict(width=0.3, color='#555555'),
        opacity=0.05, hoverinfo='none', showlegend=False
    ), row=1, col=col)
    # Hedef grup
    tgt = [n for n in G_vis.nodes() if G_vis.nodes[n].get('is_misinfo') == is_mis]
    if not tgt:
        continue
    G_sub38 = G_vis.subgraph(tgt)
    es38x, es38y = [], []
    for u, v in G_sub38.edges():
        x0, y0 = pos_vis.get(u, (0,0)); x1, y1 = pos_vis.get(v, (0,0))
        es38x += [x0, x1, None]; es38y += [y0, y1, None]
    fig38.add_trace(go.Scatter(
        x=es38x, y=es38y,
        mode='lines',
        line=dict(width=0.7, color=edge_hi),
        opacity=0.3, hoverinfo='none', showlegend=False
    ), row=1, col=col)
    avg_deg_g = np.mean([Lcc.degree(n, weight='weight') for n in tgt])
    fig38.add_trace(go.Scatter(
        x=[pos_vis[n][0] for n in tgt],
        y=[pos_vis[n][1] for n in tgt],
        mode='markers',
        name='Yanlış Bilgi' if is_mis else 'Doğru Bilgi',
        marker=dict(
            size=[5 + 20 * (Lcc.degree(n, weight='weight') / max(dict(Lcc.degree(weight='weight')).values())) for n in tgt],
            color=hi_color, opacity=0.9, line=dict(width=0)
        ),
        text=[f"<b>👤 {n}</b><br>🔗 Degree: {degree_c.get(n,0):.4f}<br>⚖️ Weighted degree std: {Lcc.degree(n, weight='weight'):.2f}<br>📊 Engagement z: {Lcc.nodes[n].get('engagement_z',0):.2f}" for n in tgt],
        hovertemplate='%{text}<extra></extra>'
    ), row=1, col=col)

fig38.update_layout(
    **LAYOUT, hovermode="closest", height=650,
    title=dict(
        text="Yanlış Bilgi vs Doğru Bilgi — Ağdaki Konum Karşılaştırması<br>"
             "<sup>Gri = Tüm ağ | Renkli = İlgili grup | Boyut = Merkezi önem | Üzerine gelin → detay</sup>",
        font=dict(color=TEXT, size=14)
    ),
    xaxis=AXIS_HIDDEN, yaxis=AXIS_HIDDEN,
    xaxis2=AXIS_HIDDEN, yaxis2=AXIS_HIDDEN,
)
fig38.update_annotations(font=dict(color=TEXT, size=12))
fig38.show()
print('YORUM: Yanlış bilgi hesapları ağın merkezi bölgelerinde yoğunlaşırken doğru bilgi hesapları daha dağınık konumdadır.')


#### 9.6 Hashtag Eş-Görünüm Ağı

Aynı gönderide birlikte geçen hashtagler arasında kenar kurularak hashtag ilişki ağı oluşturulur.

In [ ]:
TagG = nx.Graph()

hashtag_col = 'hastags' if 'hastags' in df.columns else 'hashtags'

for raw_tags in df[hashtag_col]:
    tags = parse_hashtags(raw_tags)
    tags_clean = [t.lstrip('#') for t in tags]

    for i, t1 in enumerate(tags_clean):
        for t2 in tags_clean[i+1:]:
            if TagG.has_edge(t1, t2):
                TagG[t1][t2]['weight'] += 1
            else:
                TagG.add_edge(t1, t2, weight=1)

MIN_WEIGHT = 2

TagG_f = nx.Graph()
for u, v, d in TagG.edges(data=True):
    if d['weight'] >= MIN_WEIGHT:
        TagG_f.add_edge(u, v, weight=d['weight'])

print(f'Hashtag eş-görünüm ağı: {TagG_f.number_of_nodes()} düğüm, {TagG_f.number_of_edges()} kenar (ağırlık ≥ {MIN_WEIGHT})')

In [ ]:
if TagG_f.number_of_nodes() > 0:

    TOP_N = 80
    tag_degrees_all = dict(TagG_f.degree(weight='weight'))
    top_tags = sorted(tag_degrees_all, key=tag_degrees_all.get, reverse=True)[:TOP_N]
    TagG_vis = TagG_f.subgraph(top_tags).copy()

    pos_tag = nx.spring_layout(TagG_vis, k=1.2, iterations=400, seed=42)

    tag_degrees = dict(TagG_vis.degree())
    tag_weighted_degrees = dict(TagG_vis.degree(weight='weight'))

    weights = [TagG_vis[u][v]['weight'] for u, v in TagG_vis.edges()]
    max_w = max(weights) if weights else 1

    edge_x, edge_y = [], []
    for u, v in TagG_vis.edges():
        x0, y0 = pos_tag[u]
        x1, y1 = pos_tag[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode='lines',
        line=dict(width=0.6, color='#999999'),
        hoverinfo='none',
        showlegend=False,
        opacity=0.25
    )

    nodes_t = list(TagG_vis.nodes())

    max_deg = max(tag_degrees.values()) if tag_degrees else 1

    node_sizes = [
        10 + 28 * (tag_degrees[n] / max_deg) ** 0.5
        for n in nodes_t
    ]

    label_threshold = np.percentile(list(tag_degrees.values()), 85)

    hover_t = [
        f"<b>#{n}</b><br>"
        f"🔗 Derece: {tag_degrees[n]}<br>"
        f"⚖️ Ağırlıklı derece: {tag_weighted_degrees[n]}<br>"
        f"🔁 İlk bağlantılar: {', '.join(list(TagG_vis.neighbors(n))[:5])}"
        for n in nodes_t
    ]

    node_trace = go.Scatter(
        x=[pos_tag[n][0] for n in nodes_t],
        y=[pos_tag[n][1] for n in nodes_t],
        mode='markers+text',
        marker=dict(
            size=node_sizes,
            color='#0EA5E9',
            opacity=0.9,
            line=dict(width=0.8, color='white')
        ),
        text=[
            n if tag_degrees[n] >= label_threshold else ""
            for n in nodes_t
        ],
        textposition='top center',
        textfont=dict(size=10, color=TEXT),
        hovertext=hover_t,
        hovertemplate='%{hovertext}<extra></extra>',
        showlegend=False
    )

    fig_tag = go.Figure(data=[edge_trace, node_trace])

    fig_tag.update_layout(
        **LAYOUT,
        hovermode="closest",
        height=750,
        title=dict(
            text="Hashtag Eş-Görünüm Ağı<br>"
                 "<sup>En merkezi 80 hashtag | Boyut = derece | Kenar = aynı gönderide birlikte geçme</sup>",
            font=dict(color=TEXT, size=14)
        ),
        xaxis=AXIS_HIDDEN,
        yaxis=AXIS_HIDDEN
    )

    fig_tag.show()

else:
    print('Hashtag ağı için yeterli bağlantı yok.')

#### 9.7. Betweenness Merkeziliğine Göre Ölçeklendirilmiş Ağ

In [ ]:
fig43_ex, fig43_ey = [], []
nodes_vis2 = list(Lcc.nodes())
between_arr2 = np.array([betweenness_c.get(n, 0) for n in nodes_vis2])
between_norm2 = between_arr2 / (between_arr2.max() or 1)

edge_list2 = list(Lcc.edges())
edge_bw = [(betweenness_c.get(u, 0) + betweenness_c.get(v, 0))/2 for u, v in edge_list2]
max_bw  = max(edge_bw) if edge_bw else 1

for u, v, bw in zip([e[0] for e in edge_list2], [e[1] for e in edge_list2], edge_bw):
    x0, y0 = pos[u]; x1, y1 = pos[v]
    fig43_ex += [x0, x1, None]; fig43_ey += [y0, y1, None]

edge_tr43 = go.Scatter(
    x=fig43_ex, y=fig43_ey, mode='lines',
    line=dict(width=0.4, color='#555555'),
    opacity=0.12, hoverinfo='none', showlegend=False
)

hover_bt2 = [
    f"<b>👤 {n}</b><br>"
    f"📊 Betweenness: {betweenness_c.get(n, 0):.4f}<br>"
    f"📈 Normalize: {v:.3f}<br>"
    f"📱 Platform: {Lcc.nodes[n].get('platform', '?')}<br>"
    f"🏷️ Etiket: {LABEL_MAP.get(Lcc.nodes[n].get('label', ''), '?')}<br>"
    f"👁️ Görüntülenme: {fmt_num(Lcc.nodes[n].get('views'))}<br>"
    f"❤️ Beğeni: {fmt_num(Lcc.nodes[n].get('likes'))}<br>"
    f"📊 Etkileşim z: {fmt_z(Lcc.nodes[n].get('engagement_z'))}"
    for n, v in zip(nodes_vis2, between_norm2)
]
node_tr43 = go.Scatter(
    x=[pos[n][0] for n in nodes_vis2],
    y=[pos[n][1] for n in nodes_vis2],
    mode='markers',
    marker=dict(
        size=[4 + 40 * b for b in between_norm2],
        color=list(between_norm2), colorscale='YlOrRd', showscale=True,
        colorbar=dict(title=dict(text='Betweenness (normalize)', font=dict(color=TEXT)),
                      tickfont=dict(color=TEXT)),
        opacity=0.85, line=dict(width=0)
    ),
    text=hover_bt2,
    hovertemplate='%{text}<extra></extra>',
    showlegend=False
)

fig43 = go.Figure(data=[edge_tr43, node_tr43])
fig43.update_layout(
    **LAYOUT, hovermode="closest", height=700,
    title=dict(
        text="Betweenness Centrality Haritası — Köprü Hesaplar (Büyütülmüş)<br>"
             "<sup>Sarı/Kırmızı = Yüksek köprülük | Büyük düğüm = Güçlü ara aracı | Üzerine gelin → detay</sup>",
        font=dict(color=TEXT, size=14)
    ),
    xaxis=AXIS_HIDDEN, yaxis=AXIS_HIDDEN
)
fig43.show()

#### 9.8. Topluluklara Göre Ağ Grafiği

Büyük ağların tamamını çizmek okunabilir olmayabilir. Bu nedenle en yüksek dereceye sahip ilk 180 düğüm görselleştirilir.

In [ ]:
TOP_N = 180
degrees_dict = dict(Lcc.degree(weight='weight'))
selected_nodes = sorted(degrees_dict, key=degrees_dict.get, reverse=True)[:TOP_N]
H = Lcc.subgraph(selected_nodes).copy()

for n in H.nodes():
    H.nodes[n]['community'] = partition.get(n, 0)

pos_h = nx.spring_layout(H, seed=42, k=2.5, iterations=100)

comm_ids_h = [H.nodes[n].get('community', 0) for n in H.nodes()]
n_comms_h  = max(comm_ids_h) + 1 if comm_ids_h else 1

orig_degrees = dict(Lcc.degree(weight='weight'))
vals = [orig_degrees.get(n, 1) for n in H.nodes()]
min_d, max_d = min(vals), max(vals)
node_sizes_h = [
    8 + 40 * (orig_degrees.get(n, 1) - min_d) / (max_d - min_d + 1)
    for n in H.nodes()
]

exh, eyh = [], []
for u, v in H.edges():
    x0, y0 = pos_h[u]; x1, y1 = pos_h[v]
    exh += [x0, x1, None]; eyh += [y0, y1, None]
edge_trh = go.Scatter(
    x=exh, y=eyh, mode='lines',
    line=dict(width=0.5, color='#555555'),
    opacity=0.2, hoverinfo='none', showlegend=False
)

comm_traces_h = [edge_trh]
COMM_COLORS_H = [ACCENT, ACCENT2, "#10B981", "#F59E0B", "#8B5CF6", "#06B6D4", "#EC4899", "#14B8A6"]
unique_comms_h = sorted(set(comm_ids_h))
nodes_h_list = list(H.nodes())

for cid in unique_comms_h:
    mbrs = [n for n in nodes_h_list if H.nodes[n].get("community", 0) == cid]
    color_h = COMM_COLORS_H[cid % len(COMM_COLORS_H)]
    hover_h = [
        f"<b>👤 {n}</b><br>"
        f"🏘️ Topluluk {cid}<br>"
        f"⚖️ Std. ağırlıklı derece: {orig_degrees.get(n,0):.2f}<br>"
        f"📱 {H.nodes[n].get('platform','?')}<br>"
        f"🏷️ {LABEL_MAP.get(H.nodes[n].get('label',''),'?')}"
        for n in mbrs
    ]
    comm_traces_h.append(go.Scatter(
        x=[pos_h[n][0] for n in mbrs],
        y=[pos_h[n][1] for n in mbrs],
        mode='markers',
        name=f"Topluluk {cid} (n={len(mbrs)})",
        marker=dict(
            size=[8 + 40 * (orig_degrees.get(n,1)-min_d)/(max_d-min_d+1) for n in mbrs],
            color=color_h, opacity=0.9, line=dict(width=0.5, color='white')
        ),
        text=hover_h,
        hovertemplate='%{text}<extra></extra>'
    ))

fig45 = go.Figure(data=comm_traces_h)
fig45.update_layout(
    **LAYOUT, hovermode="closest", height=800,
    title=dict(
        text="Topluluklara Göre Renklendirilmiş Ağ (İlk 180 Merkezi Düğüm)<br>"
             "<sup>Renk = Topluluk | Boyut = Standartlaştırılmış ağırlıklı derece | Üzerine gelin → detay</sup>",
        font=dict(color=TEXT, size=14)
    ),
    xaxis=AXIS_HIDDEN, yaxis=AXIS_HIDDEN
)
fig45.show()


### 10. Dayanıklılık Analizi

**Senaryolar:**
1. En yüksek degree'li düğüm kaldırıldığında ne olur?
2. En yüksek betweenness'li düğüm kaldırıldığında ağ parçalanır mı?
3. Hedefli kaldırma vs. rastgele kaldırma karşılaştırması

In [ ]:
def simulate_removal(G, removal_order, steps=20):
    """Adım adım düğüm kaldırarak ağın parçalanmasını ölçer."""
    results = []
    G_temp = G.copy()
    total = len(removal_order)
    step_size = max(1, total // steps)
    for i in range(0, total, step_size):
        frac = i / total
        if G_temp.number_of_nodes() == 0:
            results.append((frac, 0, 0)); continue
        largest  = max(nx.connected_components(G_temp), key=len)
        lcc_frac = len(largest) / G.number_of_nodes()
        n_comps  = nx.number_connected_components(G_temp)
        results.append((frac, lcc_frac, n_comps))
        for node in removal_order[i:i+step_size]:
            if node in G_temp:
                G_temp.remove_node(node)
    return results

G_main = Lcc.copy()
degree_c_main      = {n: G_main.degree(n, weight='weight') for n in G_main.nodes()}
betweenness_c_main = bet_cen
sorted_by_degree  = sorted(G_main.nodes(), key=lambda n: degree_c_main[n], reverse=True)
sorted_by_between = sorted(G_main.nodes(), key=lambda n: betweenness_c_main[n], reverse=True)
random_order2     = list(G_main.nodes()); np.random.shuffle(random_order2)

res_degree  = simulate_removal(G_main, sorted_by_degree)
res_between = simulate_removal(G_main, sorted_by_between)
res_random  = simulate_removal(G_main, random_order2)

from plotly.subplots import make_subplots
fig49 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["En Büyük Bileşen Çöküşü", "Parçalanma (Bileşen Sayısı)"],
    horizontal_spacing=0.12
)

for (res, label, color, dash) in [
    (res_degree,  'Hedefli (Std. Weighted Degree)',      ACCENT2,    'solid'),
    (res_between, 'Hedefli (Betweenness)', ACCENT,     'dash'),
    (res_random,  'Rastgele',              '#10B981',  'dot'),
]:
    fracs, lcc_fracs, comps = zip(*res)
    fig49.add_trace(go.Scatter(
        x=list(fracs), y=list(lcc_fracs),
        mode='lines+markers', name=label,
        line=dict(color=color, width=2.5, dash=dash),
        marker=dict(size=4), legendgroup=label
    ), row=1, col=1)
    fig49.add_trace(go.Scatter(
        x=list(fracs), y=list(comps),
        mode='lines+markers', name=label,
        line=dict(color=color, width=2.5, dash=dash),
        marker=dict(size=4), legendgroup=label, showlegend=False
    ), row=1, col=2)

fig49.update_layout(
    **LAYOUT, height=520,
    title=dict(
        text="Ağ Dayanıklılık Analizi — Farklı Saldırı Stratejileri Karşılaştırması<br>"
             "<sup>Üzerine gelin → değer detayı</sup>",
        font=dict(color=TEXT, size=14)
    ),
    hovermode="x unified"
)
fig49.update_xaxes(title_text='Kaldırılan Düğüm Oranı', row=1, col=1, **AXIS_STYLE)
fig49.update_yaxes(title_text='En Büyük Bileşen Oranı (LCC)', row=1, col=1, **AXIS_STYLE)
fig49.update_xaxes(title_text='Kaldırılan Düğüm Oranı', row=1, col=2, **AXIS_STYLE)
fig49.update_yaxes(title_text='Bileşen Sayısı', row=1, col=2, **AXIS_STYLE)
fig49.update_annotations(font=dict(color=TEXT, size=11))
fig49.show()
print('YORUM: Hedefli saldırı ağı rastgele saldırıya kıyasla çok daha hızlı parçalamaktadır.')


In [ ]:
N_original = Lcc.number_of_nodes()

def get_lcc_size(graph, n_original):
    """Kalan graf içindeki en büyük bağlı bileşenin orijinal LCC'ye oranını döndürür."""
    if graph.number_of_nodes() == 0:
        return 0
    comps = list(nx.connected_components(graph))
    if len(comps) == 0:
        return 0
    return len(max(comps, key=len)) / n_original

# Kaldırma sıralarını tanımla
weighted_degree_c = {n: Lcc.degree(n, weight='weight') for n in Lcc.nodes()}
targeted_order = [n for n, _ in sorted(weighted_degree_c.items(), key=lambda x: -x[1])]
betw_order     = [n for n, _ in sorted(bet_cen.items(), key=lambda x: -x[1])]
random_order   = list(np.random.permutation(list(Lcc.nodes())))
np.random.seed(42)

# simulate_removal ile hesapla
res_targeted = simulate_removal(Lcc, targeted_order)
res_betw     = simulate_removal(Lcc, betw_order)
res_random2  = simulate_removal(Lcc, random_order)

targeted_lcc = [lcc for _, lcc, _ in res_targeted]
betw_lcc     = [lcc for _, lcc, _ in res_betw]
random_lcc   = [lcc for _, lcc, _ in res_random2]
removal_pct  = [frac * 100 for frac, _, _ in res_targeted]

fig48 = go.Figure()

fig48.add_trace(go.Scatter(
    x=removal_pct, y=targeted_lcc,
    mode='lines+markers',
    name='🎯 Hedefli (Std. Weighted Degree)',
    line=dict(color=ACCENT2, width=2.5),
    marker=dict(size=5, symbol='circle')
))

fig48.add_trace(go.Scatter(
    x=removal_pct, y=betw_lcc,
    mode='lines+markers',
    name='🌉 Betweenness Bazlı',
    line=dict(color='#F59E0B', width=2.5, dash='dash'),
    marker=dict(size=5, symbol='square')
))

fig48.add_trace(go.Scatter(
    x=removal_pct, y=random_lcc,
    mode='lines+markers',
    name='🎲 Rastgele',
    line=dict(color='#10B981', width=2.5, dash='dot'),
    marker=dict(size=5, symbol='triangle-up')
))

fig48.add_hline(
    y=0.5,
    line_dash='dot',
    line_color=BORDER,
    annotation_text='Kritik Eşik (%50 LCC)',
    annotation_font_color=TEXT
)

fig48.add_trace(go.Scatter(
    x=removal_pct + removal_pct[::-1],
    y=targeted_lcc + random_lcc[::-1],
    fill='toself',
    fillcolor='rgba(239,68,68,0.07)',
    line=dict(color='rgba(255,255,255,0)'),
    showlegend=True,
    name='Hedefli–Rastgele Fark',
    hoverinfo='skip'
))

fig48.update_layout(
    **LAYOUT,
    height=550,
    title=dict(
        text="Ağ Dayanıklılık Analizi: Hedefli vs. Rastgele Düğüm Kaldırma<br>"
             "<sup>Her adımda belirli sayıda düğüm kaldırılarak LCC'nin küçülme oranı ölçülmüştür</sup>",
        font=dict(color=TEXT, size=14)
    ),
    xaxis=dict(**AXIS_STYLE, title='Kaldırılan Düğüm Oranı (%)'),
    yaxis=dict(**AXIS_STYLE, title='En Büyük Bileşen Oranı (LCC)'),
    hovermode="x unified"
)

fig48.show()

print('✅ Dayanıklılık analizi tamamlandı.')
print(f'Başlangıç LCC düğüm sayısı: {N_original}')
print(f'simulate_removal adım sayısı: {len(removal_pct)}')
print(f'Maksimum kaldırılan düğüm oranı: %{removal_pct[-1]:.1f}')

### 11. Sonuç ve Yorumlar

Bu çalışma kapsamında İran–İsrail çatışmasıyla ilgili sosyal medya hesapları, ortak hashtag kullanımı üzerinden ağ yapısına dönüştürülmüş ve deepfake/yanlış bilgi yayılımı sosyal ağ analizi yöntemleriyle incelenmiştir. Elde edilen sonuçlar, yanlış bilgi ve deepfake içeriklerinin ağ içinde rastgele dağılmadığını; belirli hesaplar, topluluklar ve merkezi bağlantılar etrafında yoğunlaşabildiğini göstermektedir.

Analizlerde bazı hesapların yüksek degree, weighted degree, betweenness, closeness ve PageRank değerleriyle öne çıktığı görülmüştür. Bu hesaplar ağ içinde popüler, köprü, hızlı yayıcı veya otorite rolü üstlenebilecek aktörlerdir. Özellikle weighted degree ve betweenness değeri yüksek hesaplar, hem güçlü etkileşim bağlantılarına sahip olmaları hem de farklı topluluklar arasında geçiş noktası oluşturmaları nedeniyle bilgi yayılımında kritik konumdadır.

Topluluk analizi sonucunda ağın farklı gruplara ayrıldığı ve bazı topluluklarda deepfake/yanlış bilgi oranının daha yüksek olduğu belirlenmiştir. Bu durum, yanlış bilginin tüm ağda eşit dağılmadığını; belirli platform veya hesap kümelerinde daha yoğun görülebildiğini göstermektedir. Özellikle bazı TikTok ve Instagram ağırlıklı topluluklarda riskli içerik oranlarının daha yüksek olması dikkat çekmektedir.

Dayanıklılık analizi sonuçları, rastgele hesap kaldırmanın ağı daha sınırlı etkilediğini; buna karşılık merkezi veya köprü rolündeki hesapların kaldırılması durumunda ağın daha hızlı parçalandığını göstermiştir. Bu sonuç, ağın belirli kritik hesaplara duyarlı olduğunu ve bu hesapların bilgi yayılımı açısından önemli rol oynadığını ortaya koymaktadır.

Genel olarak bu çalışmada, İran–İsrail çatışması bağlamında deepfake ve yanlış bilgi yayılımının sadece içerik düzeyinde değil, ağ yapısı üzerinden de analiz edilebileceği görülmüştür. Sosyal ağ analizi sayesinde hangi hesapların daha merkezi olduğu, hangi toplulukların daha riskli içerik barındırdığı ve yanlış bilginin hangi bağlantılar üzerinden yayılabileceği ortaya konmuştur.